In [1]:
import os
#%env JAX_PLATFORMS=cpu
%env CUDA_VISIBLE_DEVICES=0
%env XLA_PYTHON_CLIENT_PREALLOCATE=false
os.chdir('..')
from tqdm.auto import tqdm
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import numpy as np
import odds_datasets

from balif import Balif
from sklearn.metrics import average_precision_score

env: CUDA_VISIBLE_DEVICES=0
env: XLA_PYTHON_CLIENT_PREALLOCATE=false


In [2]:
seed = 42
n_sims = 32

model_configs: dict = {
    "IF": dict(
        hyperplane_components=1,
        p_normal_idx="uniform",
        p_normal_value="uniform",
        p_intercept="uniform",
    ),
    "EIF": dict(
        hyperplane_components=-1,
        p_normal_idx="uniform",
        p_normal_value="uniform",
        p_intercept="uniform",
    )
}

In [3]:
@eqx.filter_jit
def run_fn(model, train_data, train_labels, test_data, key):
    def scan_body(carry, key):
        key_score, key_query, key_update = jr.split(key, 3)
        model, queriable = carry

        scores = model.score(test_data, key=key_score)

        interests = model.interest(train_data, key=key_query)
        query_idx = jnp.where(queriable, interests, interests.min()).argmax()
        queriable = queriable.at[query_idx].set(False)
        point, is_anomaly = train_data[query_idx], train_labels[query_idx]

        model = model.register(point, is_anomaly=is_anomaly, key=key_update)
        return (model, queriable), scores

    samples, dim = train_data.shape
    iterations = 1 + samples
    rng_fit, rng_steps = jr.split(key)
    model = model.fit(train_data, key=rng_fit)
    queriable = jnp.ones(samples, dtype=bool)
    _, scores = jax.lax.scan(
        scan_body, (model, queriable), jr.split(rng_steps, iterations)
    )
    return scores

In [4]:
for dataset_name in (pbar := tqdm(odds_datasets.datasets_names)):
    data, labels = odds_datasets.load(dataset_name)
    pbar.set_description(f"{dataset_name}, shape: {data.shape}")

    split = odds_datasets.load_as_train_test(dataset_name)
    data_train, data_test, labels_train, labels_test = split

    for model_name, model_config in tqdm(model_configs.items(), desc="models"):
        model = Balif(**model_config)        
        scores = run_fn(model, data_train, labels_train, data_test, jr.key(seed))
        ap = np.array([average_precision_score(labels_test, s) for s in scores])
        %timeit run_fn(model, data_train, labels_train, data_test, jr.key(seed))


  0%|          | 0/17 [00:00<?, ?it/s]

models:   0%|          | 0/2 [00:00<?, ?it/s]

7.84 ms ± 1.22 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
